# Particle tracking and the water table

**Under the table and dropping**, a soliloquy

```
To drape or to drop, that’s the question—
Whether ’tis nobler in the porous mind
To suffer the gradients of outrageous flux,
Or to take arms against a sea of heads
And, by opposing, end them.
To flow, to seep—
No more—and by a seep to say we end
The slings and arrows of stochastic rain,
The thousand natural shocks that aquifers bear—
’Tis a consummation devoutly sensed.
To flow, to seep—
To seep, perchance to plume—ay, there’s the rub:
For in that plume of dissolved mystery
What heterogeneity may come,
When we have shuffled off this Darcy coil,
Must give us pause.
There’s the respect
That makes calamity of long retardation:
For who would bear the whips and scorns of time—
Pumping stress, inverse woes, parameter shame—
The pangs of unmodeled boundaries, the law’s delay,
The insolence of numerical dispersion,
And spurns that patient merit of the grid—
When he himself might his quietus make
With but a steady state?
Who would burdens bear,
To grunt and sweat beneath a transient load,
But that the dread of something after flow—
The undiscovered zone from which no tracer
Returns unbiased—puzzles the will,
And makes us rather bear those ills we have
Than drift to others that we know not of?
Thus conscience does make cowards of us all,
And thus the native hue of resolution
Is sicklied o’er with the pale cast of doubt,
And enterprises of great pitch and moment—
Conceptual models, grand and elegant—
With this regard their currents turn awry
And lose the name of action.
Soft you now,
Fair aquifer—remember me in recharge.
```

&mdash; some LLM, doing what it does best


This example represents a seepage scenario in a transient flow system. Upper layers start out dry and gradually wet over the course of the simulation. Particles are released into the dry top layer at the beginning of the simulation. By default, dry cells become inactive, and particles cannot be tracked through inactive regions. With the Newton formulation enabled, however, dry cells remain active, so tracking can proceed. We will run the flow model first without, and then with Newton, and demonstrate some configurable behaviors PRT supports for particles in dry conditions.

First, define units.

In [ ]:
time_unit = "days"
length_unit = "meters"

Set up a time discretization with 3 stress periods: one steady-state, one
transient, and a final steady-state. The injection well will be active only in the
second period.

In [ ]:
nper = 3
tdis_pd = [
    (1, 1, 1.0),
    (300, 30, 1.1),
    (1000, 1, 1.0),
]

Set up a grid discretization: three 10m-thick layers below a 1600m top surface.

In [ ]:
import numpy as np

nlay, nrow, ncol = 3, 20, 20
Lx, Ly = 100.0, 100.0
delr, delc = Lx / ncol, Ly / nrow
top = 1600.0
thickness = 10.0
bot = np.zeros((nlay, nrow, ncol))
for k in range(nlay):
    bot[k] = top - (k + 1) * thickness

Set up a base scenario workspace and a GWF model workspace inside it.

In [ ]:
from pathlib import Path

example_name = "prt-wt"
base_ws = Path("models") / example_name
gwf_name = f"{example_name}-gwf"
gwf_ws = base_ws / "gwf"
gwf_ws.mkdir(exist_ok=True, parents=True)

Set up the flow model.

In [ ]:
import flopy

gwf_sim = flopy.mf6.MFSimulation(
    sim_name=gwf_name,
    exe_name="mf6",
    version="mf6",
    sim_ws=gwf_ws,
)
gwf_tdis = flopy.mf6.ModflowTdis(
    gwf_sim,
    time_units=time_unit,
    nper=nper,
    perioddata=tdis_pd,
)
gwf = flopy.mf6.ModflowGwf(
    gwf_sim,
    modelname=gwf_name,
    # options="NEWTON",  # TODO uncomment after first pass
    save_flows=True,
)
gwf_dis = flopy.mf6.ModflowGwfdis(
    gwf,
    length_units=length_unit,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top=top,
    botm=bot,
)
gwf_ic = flopy.mf6.ModflowGwfic(gwf, strt=top)

The model grid's cells are convertible, with greater hydraulic conductivity in
the lateral than the vertical direction.

In [ ]:
npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    icelltype=1,
    k=0.5,
    k33=0.1,
    save_specific_discharge=True,
    save_saturation=True,
    save_flows=True,
)

Configure STO convertibility to match NPF and steady-state/transient options to match
the time discretization.

In [ ]:
sto = flopy.mf6.ModflowGwfsto(
    gwf,
    iconvert=1,
    ss=0.0001,
    sy=0.1,
    steady_state={0: True, 2: True},
    transient={1: True},
    save_flows=True,
)

Constant-head boundaries establish a mild head gradient from upper-left to lower-right.

In [ ]:
chd0_cellid = (1, 0, 0)
chd1_cellid = (1, nrow - 1, ncol - 1)
chd = flopy.mf6.ModflowGwfchd(
    gwf,
    stress_period_data={
        i: [(*chd0_cellid, 1587.0), (*chd1_cellid, 1582.0)] for i in range(nper)
    },
    save_flows=True,
)

An injection well sits in the middle layer near the upper-left quadrant. The well is active only in the transient period, with rate 100 m³/d.

In [ ]:
wel_cellid = (1, nrow // 4, ncol // 4)
well = flopy.mf6.ModflowGwfwel(
    gwf,
    stress_period_data={i: [(*wel_cellid, 100.0)] for i in range(1, nper)},
    save_flows=True,
)

Set up the output control package.

In [ ]:
oc = flopy.mf6.ModflowGwfoc(
    gwf,
    budget_filerecord=f"{gwf_name}.cbb",
    head_filerecord=f"{gwf_name}.hds",
    saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
)

Set up the iterative model solution. Solver settings are more complex when Newton is on.

In [ ]:
ims = flopy.mf6.ModflowIms(
    gwf_sim,
    complexity="MODERATE",
    outer_dvclose=1e-5,
    inner_dvclose=1e-6,
    pname=gwf_name,
)

In [ ]:
# TODO: Comment out the block above and uncomment this one when Newton is enabled.

# flopy.mf6.ModflowIms(
#     gwf_sim,
#     print_option="SUMMARY",
#     outer_dvclose=1e-5,
#     outer_maximum=100,
#     under_relaxation="DBD",
#     under_relaxation_gamma=0.01,
#     under_relaxation_theta=0.7,
#     under_relaxation_kappa=0.01,
#     under_relaxation_momentum=0.0,
#     inner_maximum=100,
#     inner_dvclose=1e-6,
#     rcloserecord=0.1,
#     linear_acceleration="BICGSTAB",
#     relaxation_factor=0.99,
#     number_orthogonalizations=2,
#     reordering_method="NONE",
#     pname=gwf_name,
# )

Now write and run the flow simulation.

In [ ]:
gwf_sim.write_simulation(silent=True)
gwf_sim.run_simulation(silent=False)

Load flow results, then plot heads in the middle layer at the end of each period.
By the end of the initial steady-state period the CHD boundaries have drained the
flow system so that the top layer is fully dry. By the end of the simulation, the
water table has risen such that the top layer is partially saturated.

In [ ]:
hds = gwf.output.head()
hds_kper0 = hds.get_data(kstpkper=(0, 0))
hds_kper1 = hds.get_data(kstpkper=(0, 1))
hds_kper2 = hds.get_data(kstpkper=(0, 2))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

with flopy.plot.styles.USGSPlot():
    fig, axes = plt.subplots(1, 3, figsize=(11, 5))
    for i, (ax, arr, title) in enumerate(
        [
            (axes[0], hds_kper0[1], "Layer 1 head, end of period 1"),
            (axes[1], hds_kper1[1], "Layer 1 head, end of period 2"),
            (axes[2], hds_kper2[1], "Layer 1 head, end of period 3"),
        ]
    ):
        ax.set_aspect("equal")
        ax.set_title(title, fontsize=10)
        if i == 0:
            ax.legend(
                handles=[
                    Patch(color="red", label="WELL"),
                    Patch(color="blue", label="CHD"),
                ],
            )
        mm = flopy.plot.PlotMapView(gwf, ax=ax, layer=1)
        mm.plot_grid(alpha=0.2)
        pc = mm.plot_array(arr, alpha=0.5, vmin=1582, vmax=1587)
        plt.colorbar(pc, ax=ax, shrink=0.25, label="Head (m)")
        mm.plot_bc("WELL", plotAll=True, kper=1, color="red")
        mm.plot_bc("CHD", plotAll=True, color="steelblue")

    fig.tight_layout()
    plt.show()

Confirm that the top layer is dry by checking that its cells have head < 1590 m.

In [ ]:
dry_layer0 = (hds_kper0[0] < bot[0]).sum()
print(
    f"Dry cells in top layer after first stress period: {dry_layer0} of {nrow * ncol}"
)

We can now begin setting up the particle tracking model. We will release particles from 9 cells in the top layer: the cell directly above the injection well, and that cell's queen-neighbors.

In [ ]:
offsets = [(-1, 0, 0), (-1, -1, 0), (-1, 1, 0), (-1, 0, -1), (-1, 0, 1)]
release_cells = [
    (wel_cellid[0] + dk, wel_cellid[1] + di, wel_cellid[2] + dj)
    for dk, di, dj in offsets
]
release_cells

To avoid the need to manually calculate release coordinates, we'll again use FloPy's MP7 release-template utilities,
then convert to a format suitable for PRT.

In [ ]:
mp7_cell_data = flopy.modpath.CellDataType(
    rowcelldivisions=1, columncelldivisions=1, layercelldivisions=1
)
release_cells_T = np.array(release_cells).T
lrcregions = [
    [
        [
            min(release_cells_T[0]),
            min(release_cells_T[1]),
            min(release_cells_T[2]),
            max(release_cells_T[0]),
            max(release_cells_T[1]),
            max(release_cells_T[2]),
        ]
    ]
]
mp7_particle_data = flopy.modpath.LRCParticleData(
    subdivisiondata=mp7_cell_data,
    lrcregions=lrcregions,
)
mp7_pg = flopy.modpath.ParticleGroupLRCTemplate(
    particledata=mp7_particle_data,
)
release_pts = list(mp7_particle_data.to_prp(gwf.modelgrid))
release_pts

Now construct the PRT model. Start by creating the simulation, time and grid discretization to match the flow model above, and a Model Input Package.

In [ ]:
import os

prt_name = f"{example_name}-prt"
prt_ws = base_ws / "prt"
prt_ws.mkdir(exist_ok=True, parents=True)

prt_sim = flopy.mf6.MFSimulation(
    sim_name=prt_name,
    exe_name="mf6",
    version="mf6",
    sim_ws=prt_ws,
)
prt_tdis = flopy.mf6.ModflowTdis(
    prt_sim,
    time_units=time_unit,
    nper=nper,
    perioddata=tdis_pd,
)
prt = flopy.mf6.ModflowPrt(
    prt_sim,
    modelname=prt_name,
    model_nam_file=f"{prt_name}.name",
)
prt_dis = flopy.mf6.ModflowPrtdis(
    prt,
    length_units=length_unit,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top=top,
    botm=bot,
)
prt_mip = flopy.mf6.ModflowPrtmip(prt, porosity=0.1)
prt_prp = flopy.mf6.ModflowPrtprp(
    prt,
    nreleasepts=len(release_pts),
    packagedata=release_pts,
    nreleasetimes=1,
    releasetimes=[(0.0,)],
    print_input=True,
    drape=True,
)
prt_oc = flopy.mf6.ModflowPrtoc(
    prt,
    budget_filerecord=[f"{prt_name}.bud"],
    track_filerecord=[f"{prt_name}.trk"],
    trackcsv_filerecord=[f"{prt_name}.csv"],
    saverecord=[("BUDGET", "ALL")],
    ntracktimes=1,
    tracktimes=[(100.0,)],
)
rel_gwf_ws = os.path.relpath(gwf_ws, prt_ws)
prt_fmi = flopy.mf6.ModflowPrtfmi(
    prt,
    packagedata=[
        ("GWFHEAD", f"{rel_gwf_ws}/{gwf_name}.hds"),
        ("GWFBUDGET", f"{rel_gwf_ws}/{gwf_name}.cbb"),
    ],
)
prt_ems = flopy.mf6.ModflowEms(prt_sim, pname="ems", filename=f"{prt_name}.ems")

Write and run the PRT model.

In [ ]:
prt_sim.write_simulation(silent=False)
prt_sim.run_simulation(silent=False)

Load and plot PRT's results.

In [ ]:
import pandas as pd

pls_prt = pd.read_csv(prt_ws / f"{prt_name}.csv")

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    ax.set_aspect("equal")
    mm = flopy.plot.PlotMapView(gwf, ax=ax, layer=0)
    mm.plot_grid(alpha=0.2, lw=0.4)
    mm.plot_array(hds_kper1[1], alpha=0.25, vmin=1582, vmax=1587)
    mm.plot_bc("WELL", plotAll=True, kper=1, color="red")
    mm.plot_bc("CHD", plotAll=True, color="steelblue")
    mm.plot_pathline(pls_prt, layer="all", alpha=0.5, linewidth=0.5)
    pc = ax.scatter(pls_prt.x, pls_prt.y, c=pls_prt.t, s=5)
    cb = plt.colorbar(pc, shrink=0.5, pad=0.1)
    cb.ax.set_xlabel(r"Time (days)")
    ax.set_title("Pathlines, points colored by travel time", fontsize=11)
    ax.legend(
        handles=[
            Patch(color="red", label="WELL"),
            Patch(color="blue", label="CHD"),
        ],
    )

    fig.tight_layout()
    plt.show()

Let's now make a 3D plot with PyVista.

In [ ]:
import pyvista as pv
from flopy.export.vtk import Vtk


def get_nn(i, j):
    """Convert a structured grid 2D cell index to a node number."""
    return i * ncol + j


# create mesh for WELL boundary condition
wel_nn = get_nn(*wel_cellid[1:])
wel_pts = gwf.modelgrid.verts[gwf.modelgrid.iverts[wel_nn]]
wel_xs, wel_ys = list(zip(*wel_pts, strict=False))
wel_mesh = pv.Box(
    bounds=[
        min(wel_xs),
        max(wel_xs),
        min(wel_ys),
        max(wel_ys),
        bot[wel_cellid[0]].max(),
        bot[wel_cellid[0] - 1].max(),
    ]
)

# create mesh for CHD boundary condition
chd_nns = [get_nn(*chd0_cellid[1:]), get_nn(*chd1_cellid[1:])]
chd0_pts = gwf.modelgrid.verts[gwf.modelgrid.iverts[chd_nns[0]]]
chd1_pts = gwf.modelgrid.verts[gwf.modelgrid.iverts[chd_nns[1]]]
chd0_xs, chd0_ys = list(zip(*chd0_pts, strict=False))
chd1_xs, chd1_ys = list(zip(*chd1_pts, strict=False))
chd_mesh = pv.Box(
    bounds=[
        min(chd0_xs),
        max(chd0_xs),
        min(chd0_ys),
        max(chd0_ys),
        bot[chd0_cellid[0]].max(),
        bot[chd0_cellid[0] - 1].max(),
    ]
).merge(
    pv.Box(
        bounds=[
            min(chd1_xs),
            max(chd1_xs),
            min(chd1_ys),
            max(chd1_ys),
            bot[chd1_cellid[0]].max(),
            bot[chd1_cellid[0] - 1].max(),
        ]
    )
)

# create meshes for model results
vtk = Vtk(model=gwf, binary=False)
vtk.add_model(gwf)
vtk.add_pathline_points(pls_prt.to_records(index=False))
gwf_mesh, prt_mesh = vtk.to_pyvista()

# create plot
axes = pv.Axes(show_actor=False, actor_scale=2.0, line_width=5)
p = pv.Plotter(window_size=[700, 700])
p.enable_anti_aliasing()
p.add_mesh(gwf_mesh, opacity=0.1, style="wireframe")
p.add_mesh(wel_mesh, color="red", opacity=0.5)
p.add_mesh(chd_mesh, color="blue", opacity=0.5)
p.add_mesh(
    prt_mesh,
    point_size=8,
    line_width=2.5,
    smooth_shading=True,
    color="blue",
    scalars="t",
    scalar_bar_args={"title": "Time (days)"},
)
p.camera.elevation -= 25
p.show()

Now, let's change some settings and observe how the results change. Below are shown the decision trees PRT uses to determine what to do at release time, i.e. before tracking begins, and at tracking time.

### Release

```mermaid
flowchart LR
    ACTIVE --> |No| DRAPE(DRAPE)
    ACTIVE{Cell active?} --> |Yes| RELEASE[Release in specified cell]:::release
    DRAPE ==> |No| TERMINATE:::terminate
    DRAPE ==> |Yes| ACTIVE_UNDER{Active under?}
    ACTIVE_UNDER --> |Yes| RELEASE_AT_TABLE[Drape to active cell]:::release
    ACTIVE_UNDER --> |No| TERMINATE[Terminate]

    classDef release stroke:#98fb98
    classDef terminate stroke:#f08080
```

### Tracking

```mermaid
flowchart LR
    ACTIVE{Cell active?} --> |No| TERMINATE{Terminate}
    ACTIVE{Cell active?} --> |Yes| PARTICLE_DRY
    PARTICLE_DRY{Particle dry?} --> |Yes| DRY_TRACKING_METHOD(DRY_TRACKING_METHOD)
    DRY_TRACKING_METHOD ==> |STOP| TERMINATE[Terminate]:::terminate
    DRY_TRACKING_METHOD ==> |DROP| CELL_DRY{Cell dry?}
    CELL_DRY --> |Yes| DROP_BOTTOM[Pass to cell bottom]:::track
    CELL_DRY --> |No| DROP_TABLE([Pass to water table])
    DRY_TRACKING_METHOD ==> |STAY| STAY[Stationary]:::track
    DROP_TABLE --> TRACK:::track
    PARTICLE_DRY --> |No| TRACK[Track normally]

    classDef track stroke:#98fb98
    classDef terminate stroke:#f08080
```